## Setup

In [ ]:
import os
import sys

# Add FleetPy to path if needed
fleetpy_path = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if fleetpy_path not in sys.path:
    sys.path.append(fleetpy_path)

# Import FleetPy modules
from src.misc.globals import *
import src.evaluation.tutorial_analysis as analysis
import src.misc.config as config
from src.misc.init_modules import load_simulation_environment
import pandas as pd

print("✅ FleetPy modules imported successfully!")

In [ ]:
# Create a dictionary with the essential parameters using global variable names
config_params = {
    # Simulation environment
    # Use immediate decisions simulation environment
    G_SIM_ENV: 'ImmediateDecisionsSimulation',

    # No max decision time for immediate decisions
    G_AR_MAX_DEC_T: 0,
    # Basic request type for testing. Always accepts the operator's offer
    G_RQ_TYP1: 'BasicRequest',

    # Network and Demand
    G_NETWORK_NAME: 'example_network',             # Basic network for testing
    G_DEMAND_NAME: 'example_demand',               # Example demand pattern
    G_ZONE_SYSTEM_NAME: 'example_zones',           # Service area zones

    # Time Settings
    G_SIM_TIME_STEP: 30,                           # Update every 30 seconds
    G_SIM_START_TIME: 0,                           # Start at midnight
    G_SIM_END_TIME: 3600,                          # Run for 1 hour

    # Operational Settings
    G_NETWORK_TYPE: 'NetworkBasicWithStore',       # Basic network with store
    G_OP_MAX_WT: 300,                              # Max wait time of 5 minutes
    G_OP_MAX_DTF: 1.4,                             # Allow 40% detour
    # Constant boarding time of 30 seconds
    G_OP_CONST_BT: 30,
    G_NR_OPERATORS: 1,                             # Number of operators
    # Value of time function for vehicle routing control
    G_OP_VR_CTRL_F: 'func_key:distance_and_user_times_with_walk;vot:0.45',

    # Simulation Settings
    G_SLAVE_CPU: 1,                                # Use 1 CPU for slave processes
    "log_level": "info",                           # Set log level to INFO
    # Use a fixed random seed for reproducibility
    G_RANDOM_SEED: 0
}

# Convert to DataFrame for CSV format
config_df = pd.DataFrame(list(config_params.items()),
                         columns=['Parameter', 'Value'])

# Save constant config
config_dir = 'scenarios'
os.makedirs(config_dir, exist_ok=True)
constant_config_path = os.path.join(config_dir, 'custom_constant_config.csv')
config_df.to_csv(constant_config_path, index=False)

# Create a minimal scenario config (can override constant config values)
scenario_params = {
    G_STUDY_NAME: 'game_study',                    # Name of the study
    G_SCENARIO_NAME: 'example_pool_irsonly_sc_1',  # Scenario name
    G_RQ_FILE: "example_100.csv",                  # Example request file
    G_OP_FLEET: "default_vehtype:10",              # Size of the fleet
    # Operational module for pooling with IRS
    G_OP_MODULE: 'PoolingIRSOnly',
    G_OP_REPO_M: 'GameRepositioning' # Our algorithm
}
scenario_df = pd.DataFrame([scenario_params])
scenario_path = os.path.join(config_dir, 'scenario_custom_config.csv')
scenario_df.to_csv(scenario_path, index=False)

print("✅ Custom configuration files created!")
print("\n📝 Current configuration:")
for param, value in config_params.items():
    print(f"  - {param:.<30} {value}")

## Run Simulation

In [ ]:
# Set up paths to our custom config files
scs_path = os.path.join(os.getcwd(), 'scenarios')
constant_config_file = os.path.join(scs_path, 'custom_constant_config.csv')
scenario_file = os.path.join(scs_path, 'scenario_custom_config.csv')

# Read configuration files
constant_cfg = config.ConstantConfig(constant_config_file)
scenario_cfgs = config.ScenarioConfig(scenario_file)

# Combine configurations
scenario_cfg = constant_cfg + scenario_cfgs[0]

# Initialize simulation
sim = load_simulation_environment(scenario_cfg)

# Run the simulation
sim.run()

## Analysis

In [ ]:
# Define the results directory based on the scenario name
results_dir = os.path.join(os.getcwd(), 'results',
                           scenario_cfg[G_SCENARIO_NAME])

In [ ]:
# Get KPI summary
kpi_summary = analysis.analyze_kpis(results_dir)

# Display KPIs
print('📈 Simulation KPIs:')
print(kpi_summary.to_string(index=False))

In [ ]:
# Analyze user statistics
fig, wait_time_stats = analysis.analyze_user_stats(results_dir)
fig.show()

print('\n⏱️ Wait Time Statistics (minutes):')
print(wait_time_stats.to_string())